# Strike Extraction

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tritonoa.data.reader import read_inventory
from tritonoa.data.time import TIME_PRECISION
from scipy.signal import find_peaks

from vineyard.config import get_path

In [ ]:
sensors = ["3dvha", "vla1", "vla2"]
sensor_channels = [7, 3, 0]
time_start = np.datetime64("2023-12-01T21:51:15.00", TIME_PRECISION)
time_end = np.datetime64("2023-12-01T22:26:00.00", TIME_PRECISION)

data_streams, templates = [], []
for sensor, channel in zip(sensors, sensor_channels):
    data_streams.append(
        read_inventory(
            get_path(f"{sensor}_inventory"),
            channels=channel,
            time_start=time_start,
            time_end=time_end,
        ).taper(max_percentage=1e-4).decimate(20).filter("bandpass", [100.0, 300.0])
    )
    print(f"Sampling rate: {data_streams[-1].stats.sampling_rate} Hz")

In [ ]:
thresholds = [0.05, 0.05, 0.02]

fig, axs = plt.subplots(nrows=3, figsize=(10, 8), sharex=True)
for ax, ds, threshold in zip(axs, data_streams, thresholds):
    distance = 1.0 * ds.stats.sampling_rate
    cf = ds.data[0] ** 2
    cf /= np.max(cf)
    peaks = find_peaks(cf, height=threshold, distance=distance)[0]

    ax.plot(ds.time_vector, cf, label="Signal")
    ax.plot(ds.time_vector[peaks], cf[peaks], "ro", label="Peaks")
    ax.axhline(threshold, color="red", linestyle="--", label="Threshold")
    ax.set_title(f"Num detections: {len(peaks)}\nFirst detection: {ds.time_vector[peaks[0]]}")

plt.tight_layout()
plt.show()